In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from collections import Counter
import pickle, os

# Load full processed data (all 39 features)
X_train_full = pd.read_csv("../data-pipeline/data/processed/X_train.csv")
X_test_full  = pd.read_csv("../data-pipeline/data/processed/X_test.csv")
y_train = pd.read_csv("../data-pipeline/data/processed/y_train.csv")["class"]
y_test  = pd.read_csv("../data-pipeline/data/processed/y_test.csv")["class"]

with open("../data-pipeline/data/processed/minimal_feature_set.pkl", "rb") as f:
    features_7 = pickle.load(f)

with open("../data-pipeline/data/processed/feature_cols.pkl", "rb") as f:
    all_features = pickle.load(f)

print(f"Training rows: {len(X_train_full):,}")
print(f"Test rows:     {len(X_test_full):,}")
print(f"All features:  {len(all_features)}")
print(f"7-feature set: {features_7}")

# Sample 100K rows for EFS speed
SAMPLE = 100_000
X_s = X_train_full.sample(n=SAMPLE, random_state=42)
y_s = y_train.loc[X_s.index]

print("Running EFS for 15-feature set (takes 2-5 mins)...")

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_s, y_s)
rf_top = pd.Series(rf.feature_importances_,
                   index=all_features).nlargest(15).index.tolist()

chi_sel = SelectKBest(chi2, k=15).fit(X_s, y_s)
chi_top = X_train_full.columns[chi_sel.get_support()].tolist()

mi = mutual_info_classif(X_s, y_s, random_state=42, n_jobs=-1)
mi_top = pd.Series(mi, index=all_features).nlargest(15).index.tolist()

votes = Counter(rf_top + chi_top + mi_top)
features_15 = sorted(
    [f for f, c in votes.items() if c >= 1],
    key=lambda f: -votes[f]
)[:15]

print(f"15-feature set: {features_15}")
print(f"Full set: all {len(all_features)} features")
print("Cell 1 complete.")


Training rows: 1,916,259
Test rows:     307,727
All features:  39
7-feature set: ['Header_Length', 'Number', 'TCP', 'ack_flag_number', 'Tot size', 'ack_count', 'AVG']
Running EFS for 15-feature set (takes 2-5 mins)...
15-feature set: ['Header_Length', 'Number', 'TCP', 'ack_flag_number', 'Tot size', 'ack_count', 'AVG', 'Tot sum', 'IAT', 'HTTPS', 'Rate', 'Max', 'syn_flag_number', 'Std', 'Time_To_Live']
Full set: all 39 features
Cell 1 complete.


In [3]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

print(f"Class encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

def train_and_evaluate(feature_set, label):
    print(f"Training: {label} ({len(feature_set)} features)...")
    X_tr = X_train_full[feature_set].values
    X_te = X_test_full[feature_set].values

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(len(feature_set),)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(8,  activation='relu'),
        tf.keras.layers.Dense(3,  activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.fit(
        X_tr, y_train_enc,
        epochs=50, batch_size=256,
        validation_split=0.1,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            patience=5, restore_best_weights=True)],
        verbose=0  # silent training
    )
    y_pred = np.argmax(model.predict(X_te), axis=1)
    f1_w = f1_score(y_test_enc, y_pred, average='weighted')
    report = classification_report(
        y_test_enc, y_pred,
        target_names=le.classes_, output_dict=True
    )
    print(classification_report(y_test_enc, y_pred,
          target_names=le.classes_))
    return model, f1_w, report

print("Training function defined. Ready for Cell 3.")

Class encoding: {'Benign': 0, 'DDoS': 1, 'Reconnaissance': 2}
Training function defined. Ready for Cell 3.


In [4]:
# Run all three experiments in sequence
results = {}

model_7, f1_7, report_7 = train_and_evaluate(
    features_7, "7 features (EFS minimal set)"
)
results['7']  = {'f1': f1_7,  'report': report_7,  'features': features_7}

model_15, f1_15, report_15 = train_and_evaluate(
    features_15, "15 features (EFS extended set)"
)
results['15'] = {'f1': f1_15, 'report': report_15, 'features': features_15}

model_39, f1_39, report_39 = train_and_evaluate(
    all_features, "39 features (full set)"
)
results['39'] = {'f1': f1_39, 'report': report_39, 'features': all_features}

print("All three experiments complete.")

Training: 7 features (EFS minimal set) (7 features)...




9617/9617 [==============================] - 28s 3ms/step
                precision    recall  f1-score   support

        Benign       0.94      0.90      0.92    131582
          DDoS       1.00      1.00      1.00    159688
Reconnaissance       0.41      0.55      0.47     16457

      accuracy                           0.93    307727
     macro avg       0.78      0.82      0.80    307727
  weighted avg       0.94      0.93      0.94    307727

Training: 15 features (EFS extended set) (15 features)...
9617/9617 [==============================] - 15s 2ms/step
                precision    recall  f1-score   support

        Benign       0.96      0.86      0.90    131582
          DDoS       1.00      1.00      1.00    159688
Reconnaissance       0.37      0.68      0.48     16457

      accuracy                           0.92    307727
     macro avg       0.78      0.85      0.80    307727
  weighted avg       0.95      0.9

In [5]:
print("FEATURE COUNT ABLATION STUDY — SUMMARY")
print(f"{'Set':<14} {'Weighted F1':>12} {'DDoS F1':>10} {'Recon F1':>10} {'Benign F1':>10}")
print("-" * 58)

for key, label in [('7','7  features (EFS)'),
                   ('15','15 features (EFS+)'),
                   ('39','39 features (Full)')]:
    r = results[key]['report']
    print(
        f"{label:<14} "
        f"{r['weighted avg']['f1-score']:>12.4f} "
        f"{r['DDoS']['f1-score']:>10.4f} "
        f"{r['Reconnaissance']['f1-score']:>10.4f} "
        f"{r['Benign']['f1-score']:>10.4f}"
    )

FEATURE COUNT ABLATION STUDY — SUMMARY
Set             Weighted F1    DDoS F1   Recon F1  Benign F1
----------------------------------------------------------
7  features (EFS)       0.9371     0.9999     0.4663     0.9198
15 features (EFS+)       0.9311     1.0000     0.4828     0.9037
39 features (Full)       0.9320     1.0000     0.5024     0.9032
